In [ ]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')

In [ ]:
df['Ticket Channel'].unique()

array(['Social media', 'Chat', 'Email', 'Phone'], dtype=object)

In [ ]:
# Combine subject + description
df["text"] = (
    df["Ticket Subject"].fillna("") + " " +
    df["Ticket Description"].fillna("")
)

# Encode labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Ticket Type"])


#num_labels = len(label_encoder.classes_)
import joblib
joblib.dump(
    label_encoder,
    "/content/drive/MyDrive/CustomerIntelligence/best_model/checkpoint-3815/label_encoder.pkl"

)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pickle
import os
import joblib # Import joblib

# Define the path to your model checkpoint and label encoder
model_path = '/content/drive/MyDrive/CustomerIntelligence/best_model/checkpoint-3815'
label_encoder_path = '/content/drive/MyDrive/CustomerIntelligence/best_model/checkpoint-3815/label_encoder.pkl'

# --- 1. Load the Tokenizer ---
# The tokenizer might be saved at the base model_path or within the checkpoint folder.
# Assuming it's in the checkpoint folder, if not, adjust `tokenizer_path`.
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print("Tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading tokenizer: {e}. Trying parent directory.")
    # Fallback if tokenizer is in the parent directory
    parent_model_path = '/content/drive/MyDrive/CustomerIntelligence/best_model'
    tokenizer = AutoTokenizer.from_pretrained(parent_model_path)
    print("Tokenizer loaded from parent directory.")

# --- 2. Load the Model ---
# The model is saved as `model.safetensors` inside the checkpoint folder.
model = AutoModelForSequenceClassification.from_pretrained(model_path)
print("Model loaded successfully.")

# Set the model to evaluation mode
model.eval()

# --- 3. Load the Label Encoder ---
try:
    # Use joblib.load instead of pickle.load
    label_encoder = joblib.load(label_encoder_path)
    print("Label encoder loaded successfully.")
except FileNotFoundError:
    print(f"Error: label_encoder.pkl not found at {label_encoder_path}")
    print("Please ensure the label_encoder.pkl file is in the checkpoint directory.")
    label_encoder = None # Handle case where label encoder is missing
except (pickle.UnpicklingError, joblib.externals.loky.backend.pickle_resource.PicklingError) as e: # Catch joblib's specific pickling error if any
    print(f"An UnpicklingError occurred: {e}")
    print("This often indicates a mismatch in Python or library versions (e.g., scikit-learn) between where the object was pickled and where it's being unpickled.")
    print("Please ensure the scikit-learn version is compatible or try regenerating the label_encoder.pkl file.")
    label_encoder = None
except Exception as e:
    print(f"An unexpected error occurred while loading the label encoder: {e}")
    label_encoder = None

# --- 4. Create an Inference Function ---
def classify_ticket(text, tokenizer, model, label_encoder):
    if label_encoder is None:
        return "Cannot classify: Label encoder not loaded.", 0.0 # Return 0.0 confidence for consistency

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)

    # Move inputs to the same device as the model (e.g., GPU if available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Get the predicted class probabilities
    probabilities = torch.softmax(outputs.logits, dim=1)
    predicted_class_id = torch.argmax(probabilities, dim=1).item()

    # Decode the predicted class ID back to a human-readable label
    predicted_label = label_encoder.inverse_transform([predicted_class_id])[0]

    return predicted_label, probabilities[0][predicted_class_id].item()


Tokenizer loaded successfully.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.
Label encoder loaded successfully.

Sample Ticket Text: My internet is not working since yesterday. I cannot connect to any website.
Predicted Category: Technical issue (Confidence: 0.23)

Sample Ticket Text: I need help installing new software on my laptop.
Predicted Category: Technical issue (Confidence: 0.23)


In [ ]:
import random

random_text_value = random.choice(df['text'].tolist())
print(f"Random text value: {random_text_value}")
predicted_label,confidence = classify_ticket(random_text_value, tokenizer, model, label_encoder)
print(f"Predicted Category: {predicted_label} (Confidence: {confidence:.2f})")

Random text value: Delivery problem I'm having an issue with the LG Washing Machine. Please assist. For more help, please do not send me any messages. Please message me immediately. Thank you for shopping with me. I've noticed that the issue occurs consistently when I use a specific feature or application on my LG Washing Machine.
Predicted Category: Technical issue (Confidence: 0.22)


In [ ]:
tp_run_id = '4e2dd00dfd424521ae86c149cc6f3db3'
res_run_id = '5939fcdaa58a48719f34036bfb461a29'
tp_model_id = 'm-4462f497539a49e4b3bef147035c62f4'
res_model_id = 'm-3a8e48a1b4d0419095dfb09c2c63593d'

In [ ]:
resolution_time_model = '/content/drive/MyDrive/CustomerIntelligence/mlruns/718547084090739768/models/m-3a8e48a1b4d0419095dfb09c2c63593d/artifacts/model.pkl'
priority_classifier = '/content/drive/MyDrive/CustomerIntelligence/mlruns/158184397920343216/models/m-4462f497539a49e4b3bef147035c62f4/artifacts/model.skops'

In [ ]:
import pandas as pd

# Date conversion
df["Date of Purchase"] = pd.to_datetime(df["Date of Purchase"])
df['First Response Time'] = pd.to_datetime(df['First Response Time'])

# Reference date (latest purchase date in dataset)
reference_date = df['First Response Time'].max()

# Days since purchase
df["Days Since Purchase"] = (
    reference_date - df["Date of Purchase"]
).dt.days

# Combine Subject + Description
df["Ticket Text"] = (
    df["Ticket Subject"].fillna("") + " " +
    df["Ticket Description"].fillna("")
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(df["Ticket Priority"])

In [ ]:
#!pip install skops
import joblib
import skops.io as sio
import warnings
import pandas as pd
import numpy as np # For creating dummy features

# Suppress potential warnings from loading models that might be due to version mismatches
warnings.filterwarnings("ignore", category=UserWarning, module="skops")
warnings.filterwarnings("ignore", category=UserWarning, module="joblib")
warnings.filterwarnings("ignore", category=FutureWarning) # For potential future numpy/pandas warnings

# Model paths provided by the user
resolution_time_model_path = '/content/drive/MyDrive/CustomerIntelligence/mlruns/718547084090739768/models/m-3a8e48a1b4d0419095dfb09c2c63593d/artifacts/model.pkl'
priority_classifier_path = '/content/drive/MyDrive/CustomerIntelligence/mlruns/158184397920343216/models/m-4462f497539a49e4b3bef147035c62f4/artifacts/model.skops'

# Load the resolution time model
try:
    resolution_time_predictor = joblib.load(resolution_time_model_path)
    print("Resolution time model loaded successfully.")
    # Check if it has a predict method
    if not hasattr(resolution_time_predictor, 'predict'):
        print("Warning: Loaded resolution time model does not have a 'predict' method.")
        resolution_time_predictor = None
except Exception as e:
    print(f"Error loading resolution time model from '{resolution_time_model_path}': {e}")
    resolution_time_predictor = None

# Load the priority classifier model
try:
    # skops.io.load ensures safe loading of scikit-learn models
    # The 'trusted' argument must be a list of strings as of skops 0.10+
    # Added full type names to trusted types as per the error message.
    priority_model = sio.load(priority_classifier_path, trusted=[
        'sklearn',
        'numpy',
        'lightgbm',
        'collections',
        'collections.OrderedDict',
        'lightgbm.basic.Booster',
        'lightgbm.sklearn.LGBMClassifier',
        'sklearn.compose._column_transformer._RemainderColsList'
    ])
    print("Priority classifier model loaded successfully.")
    # Check if it has a predict method
    if not hasattr(priority_model, 'predict'):
        print("Warning: Loaded priority classifier does not have a 'predict' method.")
        priority_model = None
except Exception as e:
    print(f"Error loading priority classifier from '{priority_classifier_path}': {e}")
    priority_model = None

# Define a function to extract features for the loaded models
def extract_features_for_prediction(input_data_df, modelused):
    # Make a copy to avoid modifying the original DataFrame passed in
    df_processed = input_data_df.copy()

    if modelused == 'ticketpriority':
        # Date conversion
        df_processed["Date of Purchase"] = pd.to_datetime(df_processed["Date of Purchase"])
        df_processed['First Response Time'] = pd.to_datetime(df_processed['First Response Time'])

        # Reference date (latest purchase date in dataset)
        reference_date = df_processed['First Response Time'].max()

        # Days since purchase
        df_processed["Days Since Purchase"] = (
            reference_date - df_processed["Date of Purchase"]
        ).dt.days

        # Combine Subject + Description
        # This creates 'Ticket Text' which is used by the ColumnTransformer in the model
        df_processed["Ticket Text"] = (
            df_processed["Ticket Subject"].fillna("") + " " +
            df_processed["Ticket Description"].fillna("")
        )
        # Drop columns that are not features or that are target/ID columns
        drop_columns = [
            "Ticket ID", # ID column
            "Customer Name", "Customer Email", # Identifier columns
            "Date of Purchase", "First Response Time", # Raw date columns after processing to Days Since Purchase
            "Ticket Subject", "Ticket Description", # Raw text columns after processing to Ticket Text
            "Ticket Status", "Resolution", # Target/Status related columns
            "Time to Resolution", # Target for resolution time model
            "Ticket Priority" # Target for priority model
            # Note: If 'Ticket Priority' was used as a feature in the resolution time model, it should not be dropped here
            # For now, assuming it's only a target.
        ]
        # Filter drop_columns to only include those present in df_processed to avoid KeyError
        drop_columns = [col for col in drop_columns if col in df_processed.columns]
        df_processed = df_processed.drop(columns=drop_columns, errors='ignore')
        return df_processed
    else: # For 'resolutiontime' model
        # Convert columns to datetime
        df_processed['First Response Time'] = pd.to_datetime(df_processed['First Response Time'], errors='coerce')
        df_processed['Time to Resolution'] = pd.to_datetime(df_processed['Time to Resolution'], errors='coerce')

        # Calculate duration
        duration = df_processed['Time to Resolution'] - df_processed['First Response Time']

        # Convert duration to hours
        df_processed['Time_to_Resolve_Hours'] = duration.dt.total_seconds() / 3600

        # Optional: Replace negative values with NaN (recommended)
        df_processed.loc[df_processed['Time_to_Resolve_Hours'] < 0, 'Time_to_Resolve_Hours'] = np.nan
        df_processed["Ticket Text"] = (
            df_processed["Ticket Subject"].fillna("") + " " +
            df_processed["Ticket Description"].fillna("")
        )
        drop_columns = [
            "Time_to_Resolve_Hours", # This is likely a target, not a feature
            # The following are not necessarily present, or might be features themselves.
            # Assuming these are helper columns or targets to be dropped.
            "Time_to_Resolve_Duration",
            "Time_to_Resolve_Duration_Hours",
            "Time to Resolution", # Raw date column
            "Ticket Subject", "Ticket Description" # Raw text columns
        ]
        # Filter drop_columns to only include those present in df_processed to avoid KeyError
        drop_columns = [col for col in drop_columns if col in df_processed.columns]
        df_processed = df_processed.drop(columns=drop_columns, errors='ignore')
        return df_processed

def predict_ticket_metrics(input_df): # Changed parameter name from text_input to input_df
    """
    Predicts resolution time and ticket priority for a given input DataFrame (e.g., a single ticket or multiple).
    Requires loaded models and a feature extraction mechanism.
    """
    if resolution_time_predictor is None and priority_model is None:
        return "No models loaded. Cannot make predictions."

    results = {}

    # Extract features using the defined function for each model
    # Pass input_df to the feature extraction functions
    resolution_features = extract_features_for_prediction(input_df.copy(), 'resolutiontime') # Pass a copy to avoid modifying original input_df
    ticket_priority_features = extract_features_for_prediction(input_df.copy(), 'ticketpriority') # Pass a copy

    if resolution_time_predictor:
        try:
            # Predict resolution time (e.g., in hours, days)
            # Ensure the input to predict is a DataFrame for ColumnTransformer compatibility
            predicted_resolution_time = resolution_time_predictor.predict(resolution_features)[0]
            results["predicted_resolution_time"] = predicted_resolution_time
        except Exception as e:
            results["predicted_resolution_time"] = f"Error during resolution time prediction: {e}"

    if priority_model:
        try:
            # Predict ticket priority (e.g., 'Low', 'Medium', 'High')
            # Ensure the input to predict is a DataFrame if the model expects it (e.g., if it uses a ColumnTransformer)
            #predicted_priority = priority_model.predict(ticket_priority_features)[0]
            #results["predicted_ticket_priority"] = predicted_priority
            pred = priority_model.predict(ticket_priority_features)[0]
            predicted_priority = label_encoder.inverse_transform([pred])[0]
            results["predicted_ticket_priority"] = predicted_priority
        except Exception as e:
            results["predicted_ticket_priority"] = f"Error during priority prediction: {e}"

    return results

#print(f"\nSample Ticket Text: {sample_ticket_text}")
# Pass a single-row DataFrame to predict_ticket_metrics
#predictions = predict_ticket_metrics()
#print(f"Predictions: {predictions}")

Resolution time model loaded successfully.
Priority classifier model loaded successfully.


In [ ]:
import random

# Select a random row from the DataFrame as a new DataFrame
random_row_df = df.sample(n=1)

# Extract the 'text' value from the random row
random_text_value = random_row_df['text'].iloc[0]

print(f"Random text value: {random_text_value}")

# Pass the random single-row DataFrame to the prediction function
predictions = predict_ticket_metrics(random_row_df)
print(f"Predictions: {predictions}")

Random text value: Product recommendation I'm having an issue with the Dell XPS. Please assist.

I can't use the {product_purchased for the "Product #" field for the "Add New Product". Please assist.

I I'm worried that the issue might be hardware-related and might require repair or replacement.
[LightGBM] [Warning] Unknown parameter: eval_metric
Predictions: {'predicted_resolution_time': np.float64(11.208128764448354), 'predicted_ticket_priority': 'Critical'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Once the installation is complete, run the Streamlit app again in a *separate* terminal or cell. Then, run the following code in a *new cell* to create a public URL for your Streamlit app. Click on the URL that appears.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)


<IPython.core.display.Javascript object>

In [ ]:
streamlit run 2_Prediction.py --server.address 0.0.0.0 --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false

In [1]:
# Install ngrok
!pip install pyngrok

# Run this in a single cell
from pyngrok import ngrok
import threading
import os
import time
import subprocess

# Set up ngrok token (get one free from ngrok.com)
NGROK_AUTH_TOKEN = "39epGmId7VDVphde2lAz4FWxwHT_24GqkQUtBfSfSFzUmDYgo"  # Replace with your token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Function to run Streamlit
def run_streamlit():
    os.system("streamlit run /content/drive/MyDrive/CustomerIntelligence/app/pages/2_Prediction.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false")

# Start Streamlit in a background thread
thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to start
time.sleep(5)

# Create ngrok tunnel
try:
    public_url = ngrok.connect(8501)
    print(f"✅ Your app is running at: {public_url}")
    print("📱 Open this URL in your browser to access the app")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you set your ngrok auth token correctly")

✅ Your app is running at: NgrokTunnel: "https://unogled-missy-hungeringly.ngrok-free.dev" -> "http://localhost:8501"
📱 Open this URL in your browser to access the app


In [ ]:
streamlit run main.py --server.address 0.0.0.0 --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false